<a href="https://colab.research.google.com/github/coreprimejio/ev-server/blob/master-qa/Self_Contained_Quiz_PDF_Generator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import json
import os
import math
import random
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon, Rectangle
from pylatex import Document, Section, Command, Figure, MiniPage, LineBreak, Enumerate
from pylatex.utils import NoEscape, bold

# --- FIGURE GENERATION FUNCTIONS ---

def create_circular_division(total_parts, shaded_parts, output_filename):
    """Generates pie charts for circular divisions."""
    if total_parts <= 0: return

    num_wholes = shaded_parts // total_parts
    remaining_shaded = shaded_parts % total_parts

    num_charts = num_wholes + (1 if remaining_shaded > 0 else 0)
    if num_charts == 0: return

    fig, axes = plt.subplots(1, num_charts, figsize=(2.5 * num_charts, 2.5))
    if num_charts == 1: axes = [axes]

    for i in range(num_wholes):
        axes[i].pie([1], colors=['royalblue'], wedgeprops={'edgecolor': 'black', 'linewidth': 1})
        axes[i].axis('equal')

    if remaining_shaded > 0:
        sizes = [1] * total_parts
        colors = ['royalblue'] * remaining_shaded + ['lightgrey'] * (total_parts - remaining_shaded)
        axes[num_wholes].pie(sizes, colors=colors, startangle=90, counterclock=False,
                            wedgeprops={'edgecolor': 'black', 'linewidth': 1})
        axes[num_wholes].axis('equal')

    plt.savefig(output_filename, format='jpg', bbox_inches='tight', dpi=100)
    plt.close()

def create_rectangular_division(total_parts, shaded_parts, shading_style, output_filename):
    """Generates a horizontally long grid for rectangular divisions."""
    if total_parts == 0: return

    # Find factors to make the rectangle wider than it is tall
    factors = []
    for i in range(1, int(math.sqrt(total_parts)) + 1):
        if total_parts % i == 0:
            factors.append((i, total_parts // i))

    # Choose the factor pair with the biggest difference (most rectangular)
    rows, cols = min(factors, key=lambda x: abs(x[0] - x[1]))
    if rows > cols: # Ensure it's horizontally long
        rows, cols = cols, rows

    fig, ax = plt.subplots(figsize=(cols * 0.6, rows * 0.6))
    indices = list(range(total_parts))
    shaded_indices = random.sample(indices, shaded_parts)

    for i in range(total_parts):
        r, c = divmod(i, cols)
        is_shaded_part = i in shaded_indices

        if shading_style == 'half' and is_shaded_part:
            # Draw two distinct half-rectangles
            half1 = Rectangle((c, rows - 1 - r), 0.5, 1, facecolor='royalblue', edgecolor='black', linewidth=1)
            half2 = Rectangle((c + 0.5, rows - 1 - r), 0.5, 1, facecolor='lightgrey', edgecolor='black', linewidth=1)
            ax.add_patch(half1)
            ax.add_patch(half2)
        else:
            color = 'royalblue' if is_shaded_part else 'lightgrey'
            rect = Rectangle((c, rows - 1 - r), 1, 1, facecolor=color, edgecolor='black', linewidth=1)
            ax.add_patch(rect)

    ax.set_xlim(0, cols)
    ax.set_ylim(0, rows)
    ax.set_aspect('equal', 'box')
    ax.axis('off')
    plt.savefig(output_filename, format='jpg', bbox_inches='tight', dpi=100)
    plt.close()

def create_triangular_division(total_parts, shaded_parts, shading_style, output_filename):
    """
    Generates an image of a large equilateral triangle divided into smaller,
    equal equilateral triangles, with a random subset being shaded.
    """
    divisions = int(math.sqrt(total_parts))
    if divisions * divisions != total_parts:
        print(f"Error: total_parts must be a perfect square. Got {total_parts}.")
        return

    if shaded_parts > total_parts:
        print(f"Error: shaded_parts cannot be greater than total_parts.")
        return

    side_length = 1.0
    height = side_length * math.sqrt(3) / 2.0
    all_triangles = []

    points = {}
    for row in range(divisions + 1):
        for col in range(row + 1):
            x = (col * side_length) - (row * side_length / 2.0)
            y = -row * height
            points[(row, col)] = (x, y)

    for row in range(divisions):
        for col in range(row + 1):
            p1 = points[(row, col)]
            p2 = points[(row + 1, col)]
            p3 = points[(row + 1, col + 1)]
            all_triangles.append([p1, p3, p2])

            if col < row:
                p1_up = points[(row, col)]
                p2_up = points[(row, col + 1)]
                p3_up = points[(row + 1, col + 1)]
                all_triangles.append([p1_up, p2_up, p3_up])

    indices = list(range(total_parts))
    shaded_indices = random.sample(indices, shaded_parts)

    fig, ax = plt.subplots(figsize=(5, 5))

    for i, vertices in enumerate(all_triangles):
        is_shaded_triangle = i in shaded_indices

        if shading_style == 'half' and is_shaded_triangle:
            p1, p2, p3 = vertices
            midpoint = ((p1[0] + p2[0]) / 2, (p1[1] + p2[1]) / 2)
            half1_vertices = [p1, midpoint, p3]
            half2_vertices = [midpoint, p2, p3]

            half1 = Polygon(half1_vertices, facecolor='royalblue', edgecolor='black', linewidth=1)
            half2 = Polygon(half2_vertices, facecolor='lightgrey', edgecolor='black', linewidth=1)
            ax.add_patch(half1)
            ax.add_patch(half2)
        else:
            color = 'royalblue' if is_shaded_triangle else 'lightgrey'
            triangle = Polygon(vertices, facecolor=color, edgecolor='black', linewidth=1)
            ax.add_patch(triangle)

    ax.set_aspect('equal', 'box')
    ax.axis('off')

    main_triangle_base_width = divisions * side_length
    main_triangle_height = divisions * height
    plt.xlim(-main_triangle_base_width / 2 - 0.1, main_triangle_base_width / 2 + 0.1)
    plt.ylim(-main_triangle_height - 0.1, 0.1)

    plt.savefig(output_filename, format='jpg', bbox_inches='tight', pad_inches=0.1, dpi=100)
    plt.close()


# --- PDF GENERATION SCRIPT ---

def generate_quiz_pdf(quiz_data, pdf_filename):
    """
    Reads a JSON object of questions and generates a single-column A4 PDF.
    """
    # 1. Setup Document
    geometry_options = {"tmargin": "1in", "lmargin": "1in"}
    # Use standard 'article' class for stability
    doc = Document(pdf_filename, documentclass='article',
                   document_options=['a4paper', '12pt'],
                   geometry_options=geometry_options)

    # Add all necessary packages to the preamble
    doc.preamble.append(Command('usepackage', 'graphicx'))
    doc.preamble.append(Command('usepackage', 'amsmath'))
    doc.preamble.append(Command('usepackage', 'enumitem'))
    doc.preamble.append(Command('usepackage', 'xcolor')) # For colored text
    # Remove paragraph indentation
    doc.preamble.append(Command('setlength', [NoEscape(r'\parindent'), '0pt']))
    # Define a custom color for question headers
    doc.preamble.append(NoEscape(r'\definecolor{questioncolor}{RGB}{40,80,150}'))

    # Add Title and Author information
    doc.append(NoEscape(r'\begin{center}'))
    doc.append(NoEscape(r'{\Huge\bfseries Fractions Quiz}\\[10pt]'))
    doc.append(NoEscape(r'{\large Class 5}'))
    doc.append(NoEscape(r'\end{center}\bigskip'))

    # Create a directory for figures
    if not os.path.exists('figures'):
        os.makedirs('figures')

    # 2. Iterate through questions and add them to the PDF
    for i, q in enumerate(quiz_data['questions']):
        # Use a colored, bold header for each question
        doc.append(NoEscape(r'\textcolor{questioncolor}{\textbf{Question ' + str(i+1) + r':}} '))
        doc.append(NoEscape(q['question']))
        doc.append(NoEscape(r'\par')) # Add a paragraph break

        # Generate and add figure if it exists
        if 'figure_desc' in q:
            fig_desc = q['figure_desc']
            fig_filename = os.path.join('figures', f'q_{i+1}.jpg')

            if fig_desc['type_of_division'] == 'circular':
                create_circular_division(fig_desc['total_parts'], fig_desc['shaded_parts'], fig_filename)
            elif fig_desc['type_of_division'] == 'rectangular':
                create_rectangular_division(
                    fig_desc['total_parts'],
                    fig_desc['shaded_parts'],
                    fig_desc.get('shading_style', 'full'), # Default to 'full'
                    fig_filename
                )
            elif fig_desc['type_of_division'] == 'triangular':
                create_triangular_division(
                    fig_desc['total_parts'],
                    fig_desc['shaded_parts'],
                    fig_desc['shading_style'],
                    fig_filename
                )

            # Add the figure to the PDF directly
            if os.path.exists(fig_filename):
                # Center the image and add it directly without a float
                doc.append(NoEscape(r'\begin{center}'))
                doc.append(NoEscape(r'\includegraphics[width=0.3\linewidth]{' + fig_filename.replace('\\', '/') + '}'))
                doc.append(NoEscape(r'\end{center}'))

        # Add options
        with doc.create(Enumerate(options=NoEscape(r'label=(\alph*)'))) as enum:
            for option in q['options']:
                enum.add_item(NoEscape(option))

        # Add a horizontal line and spacing between questions
        doc.append(NoEscape(r'\bigskip\hrule\bigskip'))

    # 3. Generate PDF
    try:
        doc.generate_pdf(clean_tex=True)
        print(f"✅ Successfully generated {pdf_filename}.pdf")
    except Exception as e:
        print(f"❌ PDF generation failed. Ensure a LaTeX distribution is installed. Error: {e}")

if __name__ == '__main__':
    # The new JSON data with corrected questions
    quiz_json_object = {
      "questions": [
        {
          "question": "What fraction of the total area of the rectangle is shaded blue?",
          "figure_desc": { "type_of_division": "rectangular", "total_parts": 10, "shaded_parts": 6, "shading_style": "half" },
          "options": [ "$\\frac{6}{10}$", "$\\frac{3}{10}$", "$\\frac{4}{10}$", "$\\frac{7}{10}$" ],
          "correct_answer": 1
        },
        {
          "question": "Which of the following fractions is the smallest?",
          "options": [ "$\\frac{5}{6}$", "$\\frac{5}{7}$", "$\\frac{5}{8}$", "$\\frac{5}{9}$" ],
          "correct_answer": 3
        },
        {
          "question": "The figure shows a triangle where some parts are half-shaded. What fraction of the large triangle's total area is shaded blue?",
          "figure_desc": { "type_of_division": "triangular", "total_parts": 25, "shaded_parts": 8, "shading_style": "half" },
          "options": [ "$\\frac{8}{25}$", "$\\frac{16}{25}$", "$\\frac{4}{25}$", "$\\frac{17}{25}$" ],
          "correct_answer": 2
        },
        {
          "question": "If a box contains 24 chocolates and $\\frac{3}{8}$ of them are dark chocolate, how many are NOT dark chocolate?",
          "options": [ "9", "12", "15", "18" ],
          "correct_answer": 2
        },
        {
          "question": "What is $3 - \\frac{2}{5}$?",
          "options": [ "$2\\frac{3}{5}$", "$2\\frac{2}{5}$", "$\\frac{1}{5}$", "$3\\frac{2}{5}$" ],
          "correct_answer": 0
        },
        {
          "question": "What fraction of the rectangle is shaded?",
          "figure_desc": { "type_of_division": "rectangular", "total_parts": 18, "shaded_parts": 11, "shading_style": "full" },
          "options": [ "$\\frac{7}{18}$", "$\\frac{11}{7}$", "$\\frac{11}{18}$", "$\\frac{18}{11}$" ],
          "correct_answer": 2
        },
        {
          "question": "Which of the following is equal to $\\frac{15}{20}$?",
          "options": [ "$\\frac{3}{4}$", "$\\frac{4}{3}$", "$\\frac{5}{4}$", "$\\frac{3}{5}$" ],
          "correct_answer": 0
        },
        {
          "question": "What is the product of $\\frac{2}{5} \\times 4$?",
          "options": [ "$\\frac{8}{20}$", "$\\frac{6}{5}$", "$\\frac{2}{20}$", "$\\frac{8}{5}$" ],
          "correct_answer": 3
        },
        {
          "question": "A baker has 36 cookies. He sells $\\frac{2}{3}$ of them. How many cookies did he sell?",
          "options": [ "12", "18", "24", "30" ],
          "correct_answer": 2
        },
        {
          "question": "A ribbon is $5\\frac{1}{2}$ meters long. Another ribbon is $3\\frac{1}{4}$ meters long. What is the total length of the two ribbons?",
          "options": [ "$8\\frac{1}{3}$ m", "$8\\frac{2}{6}$ m", "$8\\frac{3}{4}$ m", "$9$ m" ],
          "correct_answer": 2
        }
      ]
    }

    pdf_file = 'New_Fractions_Quiz'
    generate_quiz_pdf(quiz_json_object, pdf_file)

✅ Successfully generated New_Fractions_Quiz.pdf
